In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DateType
current_schema = StructType(fields = [StructField("customer_id", IntegerType()),
                                      StructField("created_timestamp", TimestampType()),
                                      StructField("customer_name", StringType()),
                                      StructField("date_of_birth", DateType()),
                                      StructField("email", StringType()),
                                      StructField("member_since", DateType()),
                                      StructField("telephone", StringType())])

In [0]:
%fs ls s3://databricks-gizmobox-dl-project/operational_data/customers/

path,name,size,modificationTime
s3://databricks-gizmobox-dl-project/operational_data/customers/customers_2024_10.json,customers_2024_10.json,4389,1789223617000
s3://databricks-gizmobox-dl-project/operational_data/customers/customers_2024_11.json,customers_2024_11.json,3327,1789223617000
s3://databricks-gizmobox-dl-project/operational_data/customers/customers_2024_12.json,customers_2024_12.json,3081,1789223617000
s3://databricks-gizmobox-dl-project/operational_data/customers/customers_2025_01.json,customers_2025_01.json,2868,1789223617000


In [0]:
customers_df = spark.readStream.format("json").schema(current_schema).load("/Volumes/gizmobox/landing/operational_data/customers/")

In [0]:
from pyspark.sql import functions as F
customers_transformed_df = customers_df.withColumn("file_path",F.col("_metadata.file_path")).withColumn("ingestion_date",F.current_timestamp())

In [0]:
streamming_query = customers_transformed_df.writeStream.format("delta").option("checkpointlocation","/Volumes/gizmobox/landing/operational_data/customers_stream_checkpoint").trigger(availableNow=True).outputMode("append").toTable("gizmobox.bronze.customer_stream")

trigger = by default 500 micro seconds

trigger (processingTime = "2 minutes") user defined any minutes

trigger (availableNow=True) process all data available as multiple batches and stops

trigger (continous="2 seconds") contionusly 


output mode -> by default append

complete 

update

In [0]:
streamming_query.stop()

In [0]:
print(streamming_query.isActive)

False


In [0]:
%sql
select * from gizmobox.bronze.customer_stream

customer_id,created_timestamp,customer_name,date_of_birth,email,member_since,telephone,file_path,ingestion_date
9179,2024-10-17T16:12:27.000Z,Richard Cox,1996-10-25,devon84@mail.com,2024-09-26,+1 6680703335,dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json,2026-10-02T11:21:14.339Z
4858,2024-10-01T00:50:29.000Z,Carla Morton,2004-06-21,joseph88@mail.com,2024-09-15,+1 8616454195,dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json,2026-10-02T11:21:14.339Z
7207,2024-10-23T22:03:08.000Z,Billy Scott,1997-03-17,christopher30@mail.com,2024-09-23,+1 5544387564,dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json,2026-10-02T11:21:14.339Z
8539,2024-10-12T06:02:27.000Z,Lori Mason,2002-11-01,stephanie7@mail.com,2024-09-12,+1 0498301620,dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json,2026-10-02T11:21:14.339Z
9706,2024-10-24T13:03:13.000Z,Jennifer Haas,2001-04-03,benjamin55@mail.com,2024-10-05,+1 4725460000,dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json,2026-10-02T11:21:14.339Z
9263,2024-10-08T22:49:25.000Z,Joseph Keller,2003-02-11,null,2024-10-04,+1 3817867756,dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json,2026-10-02T11:21:14.339Z
5028,2024-10-06T19:55:52.000Z,Jessica Harris,2004-04-19,null,2024-09-10,+1 8604009935,dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json,2026-10-02T11:21:14.339Z
9018,2024-10-18T23:24:52.000Z,William Carter,2003-09-05,james70@gmail.com,2024-10-08,+1 1448753611,dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json,2026-10-02T11:21:14.339Z
8580,2024-10-21T13:20:26.000Z,Shannon Austin,2002-03-22,john30@gmail.com,2024-10-07,+1 4594705629,dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json,2026-10-02T11:21:14.339Z
3409,2024-10-02T14:53:40.000Z,Andrew Phillips,2003-04-17,peter73@yahoo.com,2024-09-30,+1 4079273853,dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json,2026-10-02T11:21:14.339Z
